# Robust pooled training: source balance and harmonization

This notebook tests whether Voisard and Felius can be pooled responsibly. It does not treat concatenation as harmonization. The same participant-level folds are used for every strategy, and all normalization statistics are fitted inside each training fold.

Strategies:
1. `naive_global`: the existing pooled baseline.
2. `source_class_balanced_global`: equalizes the four source-by-label cells while using pooled fold normalization.
3. `source_class_balanced_per_source`: equalizes source-by-label cells and normalizes each source with training-fold statistics from that source.
4. `source_class_balanced_shape`: equalizes source-by-label cells and applies per-window channel standardization to reduce amplitude/device shift.

The decision is based on the weaker source-specific participant-level AUROC, not only the pooled average.

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import balanced_accuracy_score, brier_score_loss, f1_score, log_loss, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
INTERIM = PROJECT_ROOT / 'data' / 'interim'
MAG_PATH = PROCESSED / 'validated_acceleration_magnitude_windows_float32.npy'
METADATA_PATH = PROCESSED / 'validated_window_metadata.csv'
SPLITS_PATH = INTERIM / 'participant_splits.csv'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(4)
EPOCHS = 4
BATCH_SIZE = 128
STRATEGIES = ['naive_global', 'source_class_balanced_global', 'source_class_balanced_per_source', 'source_class_balanced_shape']
metadata = pd.read_csv(METADATA_PATH)
metadata['label_binary'] = metadata['label'].map({'healthy': 0, 'stroke': 1}).astype(int)
splits = pd.read_csv(SPLITS_PATH)
magnitude_windows = np.load(MAG_PATH, mmap_mode='r')
print('Device:', DEVICE)
print('Windows:', magnitude_windows.shape)
print(metadata.groupby(['dataset_id', 'label']).participant_key.nunique())

Device: cuda
Windows: (18511, 500, 3)
dataset_id    label  
felius_2024   healthy     34
              stroke     129
voisard_2025  healthy     72
              stroke      49
Name: participant_key, dtype: int64


In [2]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def fold_indices(fold):
    role_map = splits[splits['fold'].eq(fold)].set_index('participant_key')['role']
    roles = metadata['participant_key'].map(role_map)
    return np.flatnonzero(roles.eq('training').to_numpy()), np.flatnonzero(roles.eq('validation').to_numpy())

def pooled_stats(indices):
    total = np.zeros(3, dtype='float64'); total_sq = np.zeros(3, dtype='float64'); count = 0
    for start in range(0, len(indices), 512):
        batch = np.asarray(magnitude_windows[indices[start:start + 512]], dtype='float32')
        total += batch.sum(axis=(0, 1)); total_sq += np.square(batch).sum(axis=(0, 1)); count += batch.shape[0] * batch.shape[1]
    mean = total / count; std = np.sqrt(np.maximum(total_sq / count - mean ** 2, 1e-8))
    return mean.astype('float32'), std.astype('float32')

def source_stats(indices):
    result = {}
    frame = metadata.iloc[indices]
    for source in sorted(frame.dataset_id.unique()):
        result[source] = pooled_stats(frame.index[frame.dataset_id.eq(source)].to_numpy())
    return result

def current_class_participant_weights(indices):
    frame = metadata.iloc[indices]
    pcounts = frame.groupby('participant_key').size(); ccounts = frame.groupby('label_binary').size()
    weights = frame['participant_key'].map(1.0 / pcounts).to_numpy()
    weights *= frame['label_binary'].map(len(indices) / (2.0 * ccounts)).to_numpy()
    return (weights / weights.mean()).astype('float32')

def source_class_balanced_weights(indices):
    frame = metadata.iloc[indices].copy()
    window_counts = frame.groupby('participant_key').size()
    cell_counts = frame.groupby(['dataset_id', 'label_binary']).participant_key.nunique()
    participant_weight = frame['participant_key'].map(1.0 / window_counts).to_numpy()
    cell_weight = np.array([1.0 / cell_counts[(row.dataset_id, row.label_binary)] for row in frame.itertuples()])
    weights = participant_weight * cell_weight
    return (weights / weights.mean()).astype('float32')

print('Training-fold weighting example:')
train_indices, _ = fold_indices(0)
print(metadata.iloc[train_indices].groupby(['dataset_id', 'label']).participant_key.nunique())

Training-fold weighting example:
dataset_id    label  
felius_2024   healthy     25
              stroke     100
voisard_2025  healthy     60
              stroke      42
Name: participant_key, dtype: int64


In [3]:
class GaitDataset(Dataset):
    def __init__(self, indices, strategy, global_stats, source_norm_stats, weights=None):
        self.indices = np.asarray(indices, dtype='int64'); self.strategy = strategy
        self.global_mean, self.global_std = global_stats; self.source_norm_stats = source_norm_stats
        self.weights = np.ones(len(self.indices), dtype='float32') if weights is None else weights
    def __len__(self): return len(self.indices)
    def __getitem__(self, item):
        index = int(self.indices[item]); source = metadata.iloc[index]['dataset_id']
        signal = np.asarray(magnitude_windows[index], dtype='float32')
        if self.strategy in {'naive_global', 'source_class_balanced_global'}:
            signal = (signal - self.global_mean.reshape(1, 3)) / self.global_std.reshape(1, 3)
        elif self.strategy == 'source_class_balanced_per_source':
            mean, std = self.source_norm_stats[source]
            signal = (signal - mean.reshape(1, 3)) / std.reshape(1, 3)
        elif self.strategy == 'source_class_balanced_shape':
            signal = (signal - signal.mean(axis=0, keepdims=True)) / (signal.std(axis=0, keepdims=True) + 1e-6)
        return torch.from_numpy(signal.T.copy()), torch.tensor(float(metadata.iloc[index]['label_binary'])), torch.tensor(float(self.weights[item])), torch.tensor(index)


In [4]:
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels=16):
        super().__init__(); bottleneck = min(32, in_channels)
        self.bottleneck = nn.Conv1d(in_channels, bottleneck, 1, bias=False)
        self.branches = nn.ModuleList([nn.Conv1d(bottleneck, out_channels, 7, padding=3, bias=False), nn.Conv1d(bottleneck, out_channels, 15, padding=7, bias=False), nn.Conv1d(bottleneck, out_channels, 25, padding=12, bias=False)])
        self.pool_branch = nn.Conv1d(in_channels, out_channels, 1, bias=False)
        self.bn = nn.BatchNorm1d(out_channels * 4)
        self.residual = nn.Conv1d(in_channels, out_channels * 4, 1, bias=False) if in_channels != out_channels * 4 else nn.Identity()
    def forward(self, x):
        z = self.bottleneck(x); branches = [branch(z) for branch in self.branches]
        branches.append(self.pool_branch(nn.functional.max_pool1d(x, 3, stride=1, padding=1)))
        return nn.functional.gelu(self.bn(torch.cat(branches, dim=1)) + self.residual(x))

class InceptionCNN(nn.Module):
    def __init__(self):
        super().__init__(); self.features = nn.Sequential(InceptionBlock(3), nn.MaxPool1d(2), InceptionBlock(64), nn.AdaptiveAvgPool1d(1)); self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.30), nn.Linear(64, 1))
    def forward(self, x): return self.classifier(self.features(x)).squeeze(1)

def predict(model, loader):
    model.eval(); rows = []
    with torch.no_grad():
        for signals, _, _, indices in loader:
            probabilities = torch.sigmoid(model(signals.to(DEVICE))).cpu().numpy()
            rows.extend(zip(indices.numpy(), probabilities))
    return np.array([p for _, p in rows], dtype='float32')

def participant_frame(indices, probabilities, strategy, fold, seed):
    frame = metadata.iloc[np.asarray(indices)].copy(); frame['probability'] = probabilities
    frame = frame.groupby(['participant_key', 'dataset_id', 'label_binary'], as_index=False)['probability'].mean()
    frame['strategy'] = strategy; frame['fold'] = fold; frame['seed'] = seed
    return frame

def metric_row(frame):
    y = frame['label_binary'].to_numpy(); p = frame['probability'].to_numpy(); pred = (p >= 0.5).astype(int)
    return {'participants': len(frame), 'balanced_accuracy': balanced_accuracy_score(y, pred), 'roc_auc': roc_auc_score(y, p), 'f1': f1_score(y, pred)}

In [5]:
def train_one(fold, strategy, seed=42):
    set_seed(seed); train_indices, validation_indices = fold_indices(fold)
    global_stats = pooled_stats(train_indices); source_norm_stats = source_stats(train_indices)
    if strategy == 'naive_global': weights = current_class_participant_weights(train_indices)
    else: weights = source_class_balanced_weights(train_indices)
    train_set = GaitDataset(train_indices, strategy, global_stats, source_norm_stats, weights)
    validation_set = GaitDataset(validation_indices, strategy, global_stats, source_norm_stats)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    validation_loader = DataLoader(validation_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model = InceptionCNN().to(DEVICE); optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best_auc = -np.inf; best_state = None; patience = 2
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for signals, labels, weights_batch, _ in train_loader:
            optimizer.zero_grad(); logits = model(signals.to(DEVICE))
            loss = (nn.functional.binary_cross_entropy_with_logits(logits, labels.to(DEVICE), reduction='none') * weights_batch.to(DEVICE)).mean()
            loss.backward(); optimizer.step()
        val_probabilities = predict(model, validation_loader); val_frame = participant_frame(validation_indices, val_probabilities, strategy, fold, seed); val_metrics = metric_row(val_frame)
        if val_metrics['roc_auc'] > best_auc:
            best_auc = val_metrics['roc_auc']; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}; patience = 2
        else:
            patience -= 1
            if patience == 0: break
    model.load_state_dict(best_state)
    if strategy == 'source_class_balanced_global':
        torch.save({'model_state_dict': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 'mean': global_stats[0], 'std': global_stats[1], 'fold': fold, 'strategy': strategy, 'seed': seed}, PROCESSED / f'inception_source_class_balanced_global_fold_{fold}_seed_{seed}.pt')
    val_probabilities = predict(model, validation_loader)
    val_frame = participant_frame(validation_indices, val_probabilities, strategy, fold, seed)
    return val_frame, {'strategy': strategy, 'fold': fold, 'seed': seed, **metric_row(val_frame)}

all_frames = []; fold_rows = []
for strategy in STRATEGIES:
    for fold in range(5):
        frame, row = train_one(fold, strategy, 42); all_frames.append(frame); fold_rows.append(row)
        print(strategy, 'fold', fold, 'AUROC', round(row['roc_auc'], 3), 'balanced accuracy', round(row['balanced_accuracy'], 3))
oof = pd.concat(all_frames, ignore_index=True); fold_results = pd.DataFrame(fold_rows)
print('Completed strategies:', STRATEGIES)

naive_global fold 0 AUROC 0.929 balanced accuracy 0.817


naive_global fold 1 AUROC 0.937 balanced accuracy 0.816


naive_global fold 2 AUROC 0.976 balanced accuracy 0.847


naive_global fold 3 AUROC 0.985 balanced accuracy 0.919


naive_global fold 4 AUROC 0.995 balanced accuracy 0.886


...

source_class_balanced_shape fold 0 AUROC 0.907 balanced accuracy 0.849


source_class_balanced_shape fold 1 AUROC 0.933 balanced accuracy 0.843


source_class_balanced_shape fold 2 AUROC 0.972 balanced accuracy 0.855


source_class_balanced_shape fold 3 AUROC 0.956 balanced accuracy 0.843


source_class_balanced_shape fold 4 AUROC 0.984 balanced accuracy 0.843
Completed strategies: ['naive_global', 'source_class_balanced_global', 'source_class_balanced_per_source', 'source_class_balanced_shape']


In [6]:
summary_rows = []
for strategy in STRATEGIES:
    for scope, frame in [('pooled_all', oof[oof.strategy.eq(strategy)]), *oof[oof.strategy.eq(strategy)].groupby('dataset_id')]:
        row = metric_row(frame); row.update({'strategy': strategy, 'scope': scope}); summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
source_summary = summary[summary.scope.isin(['felius_2024', 'voisard_2025'])].copy()
ranking = source_summary.groupby('strategy').roc_auc.min().sort_values(ascending=False).rename('weaker_source_auc').reset_index()
print('Source-specific OOF summary:'); print(source_summary.round(3).to_string(index=False))
print('Ranking by weaker-source AUROC:'); print(ranking.round(3).to_string(index=False))
best_strategy = ranking.iloc[0]['strategy']; print('Selected exploratory strategy:', best_strategy)

def expected_calibration_error(frame, bins=10):
    p = frame.probability.to_numpy(); y = frame.label_binary.to_numpy(); edges = np.linspace(0, 1, bins + 1); ece = 0.0
    for lower, upper in zip(edges[:-1], edges[1:]):
        mask = (p >= lower) & (p < upper if upper < 1 else p <= upper)
        if mask.any(): ece += mask.mean() * abs(p[mask].mean() - y[mask].mean())
    return float(ece)

calibration_rows = []
best_oof = oof[oof.strategy.eq(best_strategy)]
for scope, frame in [('pooled_all', best_oof), *best_oof.groupby('dataset_id')]:
    calibration_rows.append({'strategy': best_strategy, 'scope': scope, 'participants': len(frame), 'brier': brier_score_loss(frame.label_binary, frame.probability), 'log_loss': log_loss(frame.label_binary, np.clip(frame.probability, 1e-6, 1 - 1e-6)), 'ece_10bin': expected_calibration_error(frame)})
calibration = pd.DataFrame(calibration_rows)
print('Calibration for selected strategy:'); print(calibration.round(3).to_string(index=False))

Source-specific OOF summary:
 participants  balanced_accuracy  roc_auc    f1                         strategy        scope
          163              0.830    0.916 0.874                     naive_global  felius_2024
          121              0.830    0.976 0.795                     naive_global voisard_2025
          163              0.842    0.930 0.902     source_class_balanced_global  felius_2024
          121              0.924    0.982 0.909     source_class_balanced_global voisard_2025
          163              0.827    0.911 0.899 source_class_balanced_per_source  felius_2024
          121              0.927    0.989 0.906 source_class_balanced_per_source voisard_2025
          163              0.837    0.916 0.883      source_class_balanced_shape  felius_2024
          121              0.856    0.953 0.828      source_class_balanced_shape voisard_2025
Ranking by weaker-source AUROC:
                        strategy  weaker_source_auc
    source_class_balanced_global         

In [7]:
seed_rows = []
for seed in [42, 52, 62, 72, 82]:
    frame, row = train_one(0, best_strategy, seed); row['model'] = 'robust_pooled_seed_stability'; seed_rows.append(row)
seed_results = pd.DataFrame(seed_rows)
print(seed_results.round(3).to_string(index=False))

fold_results.to_csv(PROCESSED / 'robust_pooling_strategy_fold_results.csv', index=False)
summary.to_csv(PROCESSED / 'robust_pooling_strategy_summary.csv', index=False)
oof.to_csv(PROCESSED / 'robust_pooling_strategy_oof_predictions.csv', index=False)
calibration.to_csv(PROCESSED / 'robust_pooling_calibration.csv', index=False)
seed_results.to_csv(PROCESSED / 'robust_pooling_seed_stability.csv', index=False)
print('Saved robust pooling outputs to data/processed/')

                    strategy  fold  seed  participants  balanced_accuracy  roc_auc    f1                        model
source_class_balanced_global     0    42            57              0.859    0.926 0.886 robust_pooled_seed_stability
source_class_balanced_global     0    52            57              0.794    0.931 0.824 robust_pooled_seed_stability
source_class_balanced_global     0    62            57              0.806    0.918 0.883 robust_pooled_seed_stability
source_class_balanced_global     0    72            57              0.798    0.919 0.845 robust_pooled_seed_stability
source_class_balanced_global     0    82            57              0.849    0.930 0.889 robust_pooled_seed_stability
Saved robust pooling outputs to data/processed/


## Decision gate

Use the strategy with the strongest weaker-source AUROC only if its calibration and seed range remain acceptable. If all pooled strategies are weak in one source, do not force a single pooled classifier: retain source-specific models or add an explicit domain-adaptation stage and recruit a matched independent cohort.